# Filter ZINC Molecules by Node Count

This notebook downloads the maintained ZINC CSV, counts molecular graph nodes from SMILES, and writes both thresholded and exact-size filtered datasets for later training.

This notebook downloads the maintained ZINC CSV, estimates molecular graph node size from the SMILES atom count, and saves filtered CSV copies for a user-defined size threshold.

It writes two outputs:
- one CSV with molecules whose graph size is `<= USER_DEFINED_SIZE`
- one CSV with molecules whose graph size is exactly `USER_DEFINED_SIZE`


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from rdkit import Chem

from conditional_node_field_graph_generator.notebooks import configure_notebook, download_zinc_dataset
globals().update(configure_notebook(require_nsppk=False, print_torch=False))



In [ ]:
USER_DEFINED_SIZE = 20
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
OUTPUT_DIR = ZINC_DATA_ROOT / 'filtered'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = Path(download_zinc_dataset(ZINC_DATA_ROOT))
print(f'Input ZINC CSV: {csv_path}')
print(f'Filtered outputs: {OUTPUT_DIR}')


In [ ]:
def infer_smiles_column(frame: pd.DataFrame) -> str:
    candidate_columns = [
        'smiles',
        'SMILES',
        'canonical_smiles',
        'mol',
        'molecule',
    ]
    for column in candidate_columns:
        if column in frame.columns:
            return column
    object_columns = [column for column in frame.columns if frame[column].dtype == object]
    if len(object_columns) == 1:
        return object_columns[0]
    raise ValueError(
        'Unable to infer the SMILES column automatically. '
        f'Available columns: {list(frame.columns)}'
    )


def smiles_to_node_count(smiles: str) -> int:
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return -1
    return int(mol.GetNumAtoms())


In [ ]:
zinc_df = pd.read_csv(csv_path)
smiles_column = infer_smiles_column(zinc_df)
node_count_column = 'graph_node_count'

working_df = zinc_df.copy()
working_df[node_count_column] = working_df[smiles_column].map(smiles_to_node_count)
invalid_count = int((working_df[node_count_column] < 0).sum())
valid_df = working_df.loc[working_df[node_count_column] >= 0].reset_index(drop=True)

leq_df = valid_df.loc[valid_df[node_count_column] <= USER_DEFINED_SIZE].reset_index(drop=True)
exact_df = valid_df.loc[valid_df[node_count_column] == USER_DEFINED_SIZE].reset_index(drop=True)

base_name = csv_path.stem
leq_path = OUTPUT_DIR / f'{base_name}-max-nodes-{USER_DEFINED_SIZE}.csv'
exact_path = OUTPUT_DIR / f'{base_name}-exact-nodes-{USER_DEFINED_SIZE}.csv'

leq_df.to_csv(leq_path, index=False)
exact_df.to_csv(exact_path, index=False)

print(f'SMILES column: {smiles_column}')
print(f'Original rows: {len(zinc_df):,}')
print(f'Valid SMILES rows: {len(valid_df):,}')
print(f'Invalid SMILES rows skipped: {invalid_count:,}')
print(f'Rows with node count <= {USER_DEFINED_SIZE}: {len(leq_df):,}')
print(f'Rows with node count == {USER_DEFINED_SIZE}: {len(exact_df):,}')
print(f'Saved: {leq_path}')
print(f'Saved: {exact_path}')


In [ ]:
valid_df[node_count_column].value_counts().sort_index().rename('rows').to_frame().head(20)
